In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
import os

2026-02-25 23:05:17.943035: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772060718.169263      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772060718.231029      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772060718.743760      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772060718.743822      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772060718.743826      55 computation_placer.cc:177] computation placer alr

In [ ]:
d = 3 #hilbert space dimensions

input_dim = d**2
output_dim = d**2


X_train =np.load(os.path.join(save_dir, "X_train.npy"), X_train)
Y_train =np.load(os.path.join(save_dir, "Y_train.npy"), Y_train)
X_val=np.load(os.path.join(save_dir, "X_val.npy"),   X_val)
Y_val=np.load(os.path.join(save_dir, "Y_val.npy"),   Y_val)
X_test=np.load(os.path.join(save_dir, "X_test.npy"),  X_test)
Y_test=np.load(os.path.join(save_dir, "Y_test.npy"),  Y_test)

In [7]:
d = 3 #hilbert space dimensions

input_dim = d**2
output_dim = d**2

model =  models.Sequential(name=f"QST{d}")

model.add(layers.InputLayer(input_shape = (input_dim,)))

model.add(layers.Dense(200,activation ='relu'))
model.add(layers.Dense(180,activation ='relu'))
model.add(layers.Dense(180,activation ='relu'))
model.add(layers.Dense(160,activation ='relu'))
model.add(layers.Dense(160,activation ='relu'))
model.add(layers.Dense(160,activation ='relu'))
model.add(layers.Dense(160,activation ='relu'))
model.add(layers.Dense(200,activation ='relu'))

model.add(layers.Dense(output_dim, activation = 'tanh'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/input_layer.py:27: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(
2026-02-25 23:07:22.513432: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [10]:
optimizer = tf.keras.optimizers.Nadam(
    learning_rate = 0.001,
    beta_1 = 0.9,
    beta_2 = 0.999,
    epsilon = 1e-7
)

model.compile(optimizer=optimizer,
             loss='mse',
             metrics=['mae'])

model.summary()

Model: "QST3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 200)            │         2,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 180)            │        36,180 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 180)            │        32,580 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 160)            │        28,960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 160)            │        25,760 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 160)            │        25,760 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 160)            │        25,760 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 200)            │        32,200 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 209,200 (817.19 KB)

 Trainable params: 209,200 (817.19 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
early_stop= callbacks.EarlyStopping(
    monitor='val_loss',
    patience = 200,
    restore_best_weights= True,

    verbose=2
)


checkpoint = callbacks.ModelCheckpoint(
    filepath = f"best_moodel_d{d}.keras",
    monitor = 'vals_loss',
    save_best_only = True,
    verbose = 1
)

history = model.fit(
    x_train, Y_train,
    batch_size = 100,
    epochs = 2000,
    validation_data = (X_val, Y_val),
    call_backs = [early_stop, checkpoints],
    verbose = 1
)

test_loss, test_mae  = model.evaluate(X_test, Y_test, verbose = 0)

print(f"nTest loss (MSE): {test_loss:.6f}")
print(f"Test MAE: {test_mae:.6f}")

model.save(f"final_model_d{d}.keras")